In [2]:
import torch
import torch.nn as nn
import numpy as np

In [3]:
class FeedForward(nn.Module):
    def __init__(self,d_model,d_ff):
        super().__init__()
        self.fc1=nn.Linear(d_model,d_ff)
        self.activation =nn.ReLU()
        self.fc2=nn.Linear(d_ff,d_model)

    def forward(self,x):
        x=self.fc1(x)
        x=self.activation(x)
        x=self.fc2(x)
        return x

In [16]:
class GQA(nn.Module):
    def __init__(self, d_model, num_heads, num_kv_heads):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads
        self.d_k = d_model // num_heads
        self.group_size = num_heads // num_kv_heads

        self.W_q = nn.Linear(d_model, self.d_k * num_heads)
        self.W_k = nn.Linear(d_model, self.d_k * num_kv_heads)
        self.W_v = nn.Linear(d_model, self.d_k * num_kv_heads)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, x):
        batch, seq_len, _ = x.shape

        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)

        Q = Q.view(batch, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        K = K.view(batch, seq_len, self.num_kv_heads, self.d_k).transpose(1, 2)
        V = V.view(batch, seq_len, self.num_kv_heads, self.d_k).transpose(1, 2)

        K = torch.repeat_interleave(K, self.group_size, dim=1)
        V = torch.repeat_interleave(V, self.group_size, dim=1)

        mask = torch.triu(torch.ones(seq_len, seq_len, device=x.device), diagonal=1).bool()
        scores = Q @ K.transpose(-2, -1) / (self.d_k ** 0.5)
        scores = scores.masked_fill(mask, float('-inf'))
        weights = torch.softmax(scores, dim=-1)
        attn_out = weights @ V

        output = attn_out.transpose(1, 2).reshape(batch, seq_len, self.d_model)
        output = self.W_o(output)
        return output


In [5]:
class TransfomerBlock(nn.Module):
    def __init__(self,d_model,num_heads,num_kv_heads,d_ff):
        super().__init__()
        self.attn = GQA(d_model, num_heads, num_kv_heads)
        self.ff=FeedForward(d_model,d_ff)
        self.norm1=nn.LayerNorm(d_model)
        self.norm2=nn.LayerNorm(d_model)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.ff(self.norm2(x))
        return x



In [6]:
block = TransfomerBlock(d_model=16, num_heads=4, num_kv_heads=2, d_ff=64)
x=torch.randn(6,16)
out=block(x)
print(x.shape)

torch.Size([6, 16])


In [18]:
class SmallLM(nn.Module):
    def __init__(self,vocab_size,d_model,num_heads,num_kv_heads,d_ff,num_layers,max_seq_len):
        super().__init__()
        self.token_embedding=nn.Embedding(vocab_size,d_model)
        

        self.positional_embedding=nn.Embedding(max_seq_len,d_model)
        

        self.full_block=nn.ModuleList(TransfomerBlock(d_model,num_heads,num_kv_heads,d_ff) for _ in range(num_layers))
        

        self.norm_final=nn.LayerNorm(d_model)

        self.lm_head=nn.Linear(d_model,vocab_size)


    def forward(self, token_ids):
        batch, seq_len = token_ids.shape

        token_emb = self.token_embedding(token_ids)          # (batch, seq_len, d_model)
        positions = torch.arange(seq_len, device=token_ids.device)
        pos_emb = self.positional_embedding(positions)        # (seq_len, d_model)

        x = token_emb + pos_emb   # broadcasts pos_emb across the batch dimension

        for block in self.full_block:
            x = block(x)

        x = self.norm_final(x)
        logits = self.lm_head(x)
        return logits



In [19]:
model = SmallLM(vocab_size=100, d_model=16, num_heads=4, num_kv_heads=2, d_ff=64, num_layers=2, max_seq_len=20)
token_ids = torch.randint(0, 100, (6,))
logits = model(token_ids)
print(logits.shape)  # should be (6, 100) -- 6 tokens, each a score over 100 possible vocab words


ValueError: not enough values to unpack (expected 2, got 1)

In [9]:
!pip install  -q tiktoken

In [10]:
import tiktoken

enc = tiktoken.get_encoding("gpt2")

text = "the cat sat on the mat"
ids = enc.encode(text)
print(ids)
print(enc.decode(ids))
print(enc.n_vocab)


[1169, 3797, 3332, 319, 262, 2603]
the cat sat on the mat
50257


In [11]:
%pip install -q datasets

from datasets import load_dataset

dataset = load_dataset("roneneldan/TinyStories")
print(dataset)
print(dataset["train"][0])


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


README.md:   0%|          | 0.00/1.06k [00:00<?, ?B/s]

data/train-00000-of-00004-2d5a1467fff108(…): reconstructing file:   0%|          |  0.00B /  249MB            

data/train-00000-of-00004-2d5a1467fff108(…): downloading bytes:           |  0.00B            

data/train-00001-of-00004-5852b56a2bd28f(…): reconstructing file:   0%|          |  0.00B /  248MB            

...

data/validation-00000-of-00001-869c898b5(…): reconstructing file:   0%|          |  0.00B / 9.99MB            

data/validation-00000-of-00001-869c898b5(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2119719 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/21990 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 2119719
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 21990
    })
})
{'text': 'One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.\n\nLily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."\n\nTogether, they shared the needle and sewed the button on Lily\'s shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.'}


In [27]:
eot_token=enc.eot_token

subset=dataset["train"].select(range(100000))

all_tokens=[]
for line in subset:
    ids=enc.encode(line["text"])
    all_tokens.extend(ids)
    all_tokens.append(eot_token)

all_tokens=torch.tensor(all_tokens,dtype=torch.long)
print(all_tokens.shape)
print(all_tokens[:20])

torch.Size([21951928])
tensor([ 3198,  1110,    11,   257,  1310,  2576,  3706, 20037,  1043,   257,
        17598,   287,   607,  2119,    13,  1375,  2993,   340,   373,  2408])


In [15]:
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset

block_size=128

class TokenDataset(Dataset):
    def __init__(self,tokens,block_size):
        self.tokens=tokens
        self.block_size=block_size

    def __len__(self):
        return len(self.tokens)//(self.block_size+1)


    def __getitem__(self, idx):
        start = idx * self.block_size
        chunk = self.tokens[start : start + self.block_size + 1]
        input_ids = chunk[:-1]
        target_ids = chunk[1:]
        return input_ids, target_ids

train_ds = TokenDataset(all_tokens, block_size)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)

x, y = next(iter(train_loader))
print(x.shape, y.shape)   # (32, 128) (32, 128)

torch.Size([32, 128]) torch.Size([32, 128])


In [21]:
model = SmallLM(vocab_size=50257, d_model=128, num_heads=4, num_kv_heads=2, d_ff=512, num_layers=4, max_seq_len=block_size)
x, y = next(iter(train_loader))
logits = model(x)
print(logits.shape)   # should be (32, 128, 50257)


torch.Size([32, 128, 50257])


In [25]:
from torch.optim.lr_scheduler import LambdaLR

warmup_steps = 100
total_steps = 2000  # tune to your actual planned step count

def lr_lambda(step):
    if step < warmup_steps:
        return step / warmup_steps
    return max(0.1, (total_steps - step) / (total_steps - warmup_steps))

scheduler = LambdaLR(optimizer, lr_lambda)
# call scheduler.step() right after optimizer.step() each iteration


In [26]:
import torch.nn.functional as F

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)

num_epochs = 3
model.train()

for epoch in range(num_epochs):
    for step, (x, y) in enumerate(train_loader):
        x, y = x.to(device), y.to(device)

        logits = model(x)                              # (batch, seq_len, vocab_size)
        loss = F.cross_entropy(logits.view(-1, 50257), y.view(-1))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if step % 50 == 0:
            print(f"epoch {epoch} step {step} loss {loss.item():.4f}")


epoch 0 step 0 loss 6.0095


KeyboardInterrupt: 

In [23]:
from google.colab import drive
drive.mount('/content/drive')

checkpoint_path = "/content/drive/MyDrive/small_llm_checkpoint.pt"

# inside your loop, every N steps:
torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "step": step,
}, checkpoint_path)


Mounted at /content/drive


In [29]:
num_epochs = 3
total_steps = len(train_loader) * num_epochs

warmup_steps = 100

def lr_lambda(step):
    if step < warmup_steps:
        return step / warmup_steps
    return max(0.1, (total_steps - step) / (total_steps - warmup_steps))

scheduler = LambdaLR(optimizer, lr_lambda)

model.train()
for epoch in range(num_epochs):
    for step, (x, y) in enumerate(train_loader):
        x, y = x.to(device), y.to(device)

        logits = model(x)
        loss = F.cross_entropy(logits.view(-1, 50257), y.view(-1))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()          # <-- this line was missing

        if step % 50 == 0:
            print(f"epoch {epoch} step {step} loss {loss.item():.4f} lr {scheduler.get_last_lr()[0]:.6f}")


epoch 0 step 0 loss 5.8464 lr 0.000003
epoch 0 step 50 loss 5.8026 lr 0.000153
epoch 0 step 100 loss 5.4061 lr 0.000299
epoch 1 step 0 loss 5.3649 lr 0.000296
epoch 1 step 50 loss 5.2296 lr 0.000223
epoch 1 step 100 loss 5.0013 lr 0.000150
epoch 2 step 0 loss 5.0152 lr 0.000147
epoch 2 step 50 loss 4.8785 lr 0.000074
epoch 2 step 100 loss 4.7701 lr 0.000030


In [31]:
checkpoint_path = "/content/drive/MyDrive/small_llm_checkpoint.pt"

torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "scheduler_state_dict": scheduler.state_dict(),
    "config": {
        "vocab_size": 50257,
        "d_model": 128,
        "num_heads": 4,
        "num_kv_heads": 2,
        "d_ff": 512,
        "num_layers": 4,
        "max_seq_len": block_size,
    },
    "epoch": epoch,
    "step": step,
    "loss": loss.item(),
}, checkpoint_path)

print("saved checkpoint at epoch", epoch, "step", step, "loss", loss.item())


saved checkpoint at epoch 2 step 101 loss 4.911216735839844


In [32]:
@torch.no_grad()
def generate(model, prompt, max_new_tokens=100, temperature=0.8, top_k=40):
    model.eval()
    ids = enc.encode(prompt)
    ids = torch.tensor(ids, dtype=torch.long, device=device).unsqueeze(0)  # (1, seq_len)

    for _ in range(max_new_tokens):
        ids_cond = ids[:, -block_size:]   # only keep last block_size tokens (model's max context)
        logits = model(ids_cond)          # (1, seq_len, vocab_size)
        logits = logits[:, -1, :]         # (1, vocab_size) -- only care about last position

        logits = logits / temperature
        top_vals, top_idx = torch.topk(logits, top_k)
        probs = torch.softmax(top_vals, dim=-1)
        next_token = top_idx[0, torch.multinomial(probs[0], 1)]

        ids = torch.cat([ids, next_token.view(1,1)], dim=1)

    model.train()
    return enc.decode(ids[0].tolist())

print(generate(model, "Once upon a time"))


Once upon a time, a little girl that he said, there was sad. She was very, the there was a big and his mom, there was so excited to the other. "I very happy and was so 

The time on the park. The little girl was a time, "Don. He was so happy.



One day, but, in the girl was so happy with the sky and he was so happy the little boy and he was so excited and he she had a


In [33]:
subset = dataset["train"].select(range(500000))

all_tokens = []
for line in subset:
    ids = enc.encode(line["text"])
    all_tokens.extend(ids)
    all_tokens.append(eot_token)

all_tokens = torch.tensor(all_tokens, dtype=torch.long)
print(all_tokens.shape)


torch.Size([112298425])


In [34]:
train_ds = TokenDataset(all_tokens, block_size)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
print(len(train_loader))   # total steps per epoch


27205


In [ ]:
import os

checkpoint_path = "/content/drive/MyDrive/small_llm_checkpoint_500k.pt"

start_epoch = 0
start_step = 0

if os.path.exists(checkpoint_path):
    ckpt = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    scheduler.load_state_dict(ckpt["scheduler_state_dict"])
    start_epoch = ckpt["epoch"]
    start_step = ckpt["step"] + 1
    print(f"resumed from epoch {start_epoch} step {start_step}")
else:
    print("no checkpoint found, starting fresh")

num_epochs = 1
total_steps = len(train_loader) * num_epochs
warmup_steps = 200

def lr_lambda(step):
    if step < warmup_steps:
        return step / warmup_steps
    return max(0.1, (total_steps - step) / (total_steps - warmup_steps))

scheduler = LambdaLR(optimizer, lr_lambda)

model.train()
save_every = 500

for epoch in range(start_epoch, num_epochs):
    for step, (x, y) in enumerate(train_loader):
        if epoch == start_epoch and step < start_step:
            continue   # skip already-completed steps when resuming

        x, y = x.to(device), y.to(device)

        logits = model(x)
        loss = F.cross_entropy(logits.view(-1, 50257), y.view(-1))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()

        if step % 50 == 0:
            print(f"epoch {epoch} step {step} loss {loss.item():.4f} lr {scheduler.get_last_lr()[0]:.6f}")

        if step % save_every == 0:
            torch.save({
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "config": {
                    "vocab_size": 50257, "d_model": 128, "num_heads": 4,
                    "num_kv_heads": 2, "d_ff": 512, "num_layers": 4, "max_seq_len": block_size,
                },
                "epoch": epoch,
                "step": step,
                "loss": loss.item(),
            }, checkpoint_path)


no checkpoint found, starting fresh
epoch 0 step 0 loss 4.8711 lr 0.000001
epoch 0 step 50 loss 4.8719 lr 0.000076
epoch 0 step 100 loss 4.7281 lr 0.000151
epoch 0 step 150 loss 4.6748 lr 0.000226
epoch 0 step 200 loss 4.4639 lr 0.000300
epoch 0 step 250 loss 4.5189 lr 0.000299
epoch 0 step 300 loss 4.2054 lr 0.000299
epoch 0 step 350 loss 4.3093 lr 0.000298
epoch 0 step 400 loss 4.1561 lr 0.000298
epoch 0 step 450 loss 4.2084 lr 0.000297
epoch 0 step 500 loss 4.1423 lr 0.000297
epoch 0 step 550 loss 3.8962 lr 0.000296
epoch 0 step 600 loss 3.9790 lr 0.000296
epoch 0 step 650 loss 3.8110 lr 0.000295
epoch 0 step 700 loss 3.7899 lr 0.000294
epoch 0 step 750 loss 3.8820 lr 0.000294
epoch 0 step 800 loss 3.8230 lr 0.000293
epoch 0 step 850 loss 4.0060 lr 0.000293
epoch 0 step 900 loss 3.8243 lr 0.000292
epoch 0 step 950 loss 3.6893 lr 0.000292
epoch 0 step 1000 loss 3.7170 lr 0.000291
epoch 0 step 1050 loss 3.7779 lr 0.000291
epoch 0 step 1100 loss 3.7203 lr 0.000290
epoch 0 step 1150 los

In [ ]:
print(torch.cuda.is_available())
print(device)
print(next(model.parameters()).device)